In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [30]:
from corporate_omissions.data import (
    load_trucost_raw, load_fundamentals_raw, load_gdp_raw,
    preprocess_trucost, preprocess_fundamentals, preprocess_gdp,
    load_energy_prices,
)

tr_raw = load_trucost_raw()
fu_raw = load_fundamentals_raw()
fu_raw = load_fundamentals_raw()
gdp_raw = load_gdp_raw()

tr = preprocess_trucost(tr_raw)
fu = preprocess_fundamentals(fu_raw, filter_fic=False, filter_exchg=True)
gdp = preprocess_gdp(gdp_raw)
energy = load_energy_prices()

(tr.shape, fu.shape, gdp.shape, energy.shape)

((164106, 6), (89637, 15), (15, 2), (12, 4))

In [31]:
from corporate_omissions.data import merge_panel

panel, diag = merge_panel(
    fu, tr, gdp,
    year_start=2002,
    year_end=2023,
    verbose=True
)

# Merge energy prices (year-level, like GDP)
panel = panel.merge(energy.reset_index(), on="year", how="left")
print(f"\nEnergy columns merged: {[c for c in energy.columns]}")

panel.shape

=== Merge diagnostics ===
keys: ['gvkey', 'year'] (mode=gvkey_year)
fund rows (filtered):   83,420
trucost rows (filtered): 164,106
dedup keys: ['year', 'gvkey']
after merge + filters:  71,261
duplicates dropped:     1
% rows with Trucost:    0.354
% rows with GDP:        0.994
year range:             2009–2023

Energy columns merged: ['wti_crude', 'henry_hub', 'electricity_ind', 'ppi_petroleum']


(71261, 23)

In [32]:
panel.shape

(71261, 23)

In [33]:
from corporate_omissions.utils.paths import processed_dir

OUT_PROCESSED = Path("../data/processed")
OUT_PROCESSED.mkdir(parents=True, exist_ok=True)

out = OUT_PROCESSED / "panel_merged.parquet"
panel.to_parquet(out, index=False)
out

PosixPath('../data/processed/panel_merged.parquet')